In [2]:
import cv2
import os

# Splitting videos into frames

In [3]:
# Function to convert a video into frames with a maximum frame limit

def video_to_frames(video_path, output_directory, max_frames=None):
    
    # Create the output directory if it doesn't exist
    
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    # Open the video file
    
    video_capture = cv2.VideoCapture(video_path)

    # Initialize frame count
    
    frame_count = 0

    # Read and save frames
    
    while True:
        ret, frame = video_capture.read()

        if not ret or (max_frames is not None and frame_count >= max_frames):
            break

        # Save the frame as an image
        
        frame_filename = os.path.join(output_directory, f"frame_{frame_count:04d}.jpg")
        cv2.imwrite(frame_filename, frame)

        frame_count += 1

    # Release the video capture object
    
    video_capture.release()

    print(f"Total frames extracted: {frame_count}")


In [4]:
# Example usage with a maximum of frames
video_path = r"/home/ibukhari/Documents/Dev/Signet_PersonalExperiments/datasets/Videos/_FW1JoDAW8w.mp4"
output_directory = r"datasets/Frames/_FW1JoDAW8w"
# max_frames = 10
video_to_frames(video_path, output_directory)

Total frames extracted: 1942


# Learning Dinov2

### Extracting Image Features

In [3]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

# Model name
model_name = "facebook/dinov2-small"

# 1. Load the image processor and pretrained model
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

# 2. Load an image
image = Image.open("datasets/Frames/_FW1JoDAW8w/frame_0249.jpg").convert("RGB")

# 3. Preprocess the image
inputs = processor(images=image, return_tensors="pt")

# 4. Extract visual representations
with torch.inference_mode():
    outputs = model(**inputs)
    
# 5. Inspect the result
features = outputs.last_hidden_state


# Final Print
print("\nRAW Inputs", inputs['pixel_values'].shape)
print("\nAll Output tokens:", features.shape)
print("CLS token:", features[:, 0, :].shape)
print("Patch tokens:", features[:, 1:, :].shape)

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 7319.44it/s]


RAW Inputs torch.Size([1, 3, 224, 224])

All Output tokens: torch.Size([1, 257, 384])
CLS token: torch.Size([1, 384])
Patch tokens: torch.Size([1, 256, 384])


### Extracting Video Features 
Limited to 8 sampled frames. each sample of frame is the 5th frame

In [4]:
import cv2
import torch
from transformers import AutoImageProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the pretrained DINOv2 backbone
model_name = "facebook/dinov2-small"
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# Open a sign-language video
cap = cv2.VideoCapture("/home/ibukhari/Documents/Dev/Signet_PersonalExperiments/datasets/Videos/0bodeyCThJM.mp4")

if not cap.isOpened():
    raise RuntimeError("Cannot open sign.mp4")

embeddings = []
frame_index = 0

# Extract features from up to 8 sampled frames
while len(embeddings) < 8:
    success, frame = cap.read()

    if not success:
        break

    # Keep every fifth frame
    if frame_index % 5 == 0:
        # OpenCV uses BGR; convert to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        inputs = processor(
            images=frame,
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():
            outputs = model(**inputs)

        # Extract global CLS embedding
        cls = outputs.last_hidden_state[0, 0, :]
        embeddings.append(cls.cpu())

    frame_index += 1

cap.release()

if len(embeddings) < 2:
    raise ValueError("Need at least 2 sampled frames")

video_features = torch.stack(embeddings)

print("Video features:", video_features.shape)

# Differences between consecutive embeddings
differences = torch.diff(video_features, dim=0)

print("Temporal differences:", differences.shape)


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 7346.87it/s]


Video features: torch.Size([8, 384])
Temporal differences: torch.Size([7, 384])


### temporal representation experiment

Can DINOv2 embeddings distinguish opposite movements

In [7]:

# Original sequence: [8, 384]
forward = video_features

# Reverse the temporal order
backward = torch.flip(forward, dims=[0])

# Average all frames
avg_forward = forward.mean(dim=0)
avg_backward = backward.mean(dim=0)

# comparing Averages
print(torch.allclose(avg_forward, avg_backward))



True


Would the avgs be the same if we have a difference between each consecutive frame (temporal difference)



In [13]:

# Consecutive-frame changes
delta_forward = torch.diff(forward, dim=0)
delta_backward = torch.diff(backward, dim=0)

#  we started with 8 frames
print("New shape", delta_forward.shape)
# [7, 384]

# Compare the average changes
motion_forward = delta_forward.mean(dim=0)
motion_backward = delta_backward.mean(dim=0)

# Comparing Averages
print(torch.allclose(
    motion_forward,
    motion_backward,
    atol=1e-6
))

# changing direction of 1 avg, to have same temporal direction
print(torch.allclose(
    motion_forward,
    -motion_backward,
    atol=1e-6
))


New shape torch.Size([7, 384])
False
True


The problem with averages is that it measures changes in one direction  wither + or -, it can't capture small intermediate changes.

e.g.
We move our hand from full left to full right. This might be a + direction. But it is averaging the overall direction change. What if we move our hand from full left to full right then to the middle and back to full right. The avd will only capture the overall change to full right rather than also representing the direction change in the middle. for sign language we want to capture all small changes